In [24]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

import gc
import os
from utils import agg_columns, remove_nan_columns

In [3]:
dataset_dir = "dataset"

In [4]:
csv_tables = [
    "application_train.csv",
    "bureau_balance.csv",
    "bureau.csv",
    "credit_card_balance.csv",
    "installments_payments.csv",
    "POS_CASH_balance.csv",
    "previous_application.csv",
]

(
    applications_df,
    bureau_balance_df,
    bureau_df,
    credit_card_balance_df,
    installments_payments_df,
    POS_CASH_balance_df,
    previous_application_df
) = [pd.read_csv(os.path.join(dataset_dir, table)) for table in csv_tables]

In [5]:
## Since in the paper no features capitalized on the benefits time provides, like the decay of good habit of being on time with payments, 
## we decided to try and include features with time variable in hindsight

In [6]:
def agg_numeric_columns(df, groupby_columns, metrics=["count", "mean", "sum", "min", "max"], exclude_columns=None):
    exclude_columns = exclude_columns or []

    # Find all numeric columns that are not used for grouping
    numeric_columns = df.select_dtypes(include="number").columns.tolist()
    numeric_columns = [c for c in numeric_columns if c not in groupby_columns and c not in exclude_columns]

    if not numeric_columns:
        return df[groupby_columns].drop_duplicates().reset_index(drop=True)
    
    # Compute the aggregations and return a multi-index DataFrame
    multi_index_agg = df.groupby(groupby_columns)[numeric_columns].agg(metrics)

    # Flatten the DataFrame
    agg_df = pd.DataFrame({
        f"{column}_{agg}": multi_index_agg[(column, agg)]
        for column, agg in multi_index_agg.columns
    }).reset_index()

    # Only "count"/"sum" of an empty group are 0, for "mean"/"min"/"max"
    # a missing value means "no data", so leave it as NaN to avoid misleading the model.
    fill_zero_columns = [
        f"{column}_{agg}"
        for column, agg in multi_index_agg.columns
        if agg in ("count", "sum")
    ]
    agg_df[fill_zero_columns] = agg_df[fill_zero_columns].fillna(0)

    return agg_df

In [7]:
installments_payments_df.head(15)

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585
5,1137312,164489,1.0,12,-1384.0,-1417.0,5970.375,5970.375
6,2234264,184693,4.0,11,-349.0,-352.0,29432.295,29432.295
7,1818599,111420,2.0,4,-968.0,-994.0,17862.165,17862.165
8,2723183,112102,0.0,14,-197.0,-197.0,70.740,70.740
9,1413990,109741,1.0,4,-570.0,-609.0,14308.470,14308.470


In [8]:
# This will show you how many payment rows each customer actually has
print(installments_payments_df['SK_ID_CURR'].value_counts())

SK_ID_CURR
145728    372
296205    350
453103    347
189699    344
186851    337
         ... 
411939      1
420608      1
434108      1
413433      1
434445      1
Name: count, Length: 339587, dtype: int64


In [15]:
def build_time_series_features(df):
    """
    Extracts time-aware frequency, severity, and trend metrics from the installments dataset.
    """
    print("Sorting data chronologically...")
    # 1. SORTING IS CRITICAL. We must sort by customer and due date to find the "last" installment.
    df = df.sort_values(by=['SK_ID_CURR', 'DAYS_INSTALMENT'])
    
    # 2. Base Row-Level Calculations
    # Calculate days late (positive = late, negative = early)
    df['DAYS_LATE'] = df['DAYS_ENTRY_PAYMENT'] - df['DAYS_INSTALMENT']
    
    # Apply zero-clipping: Early payments become 0 so they don't skew the severity
    df['DAYS_LATE_CLIPPED'] = df['DAYS_LATE'].clip(lower=0)
    
    # Create a binary flag: 1 if late, 0 if on time or early
    df['IS_LATE'] = (df['DAYS_LATE'] > 0).astype(int)

    df = df[df['DAYS_INSTALMENT'] < 0].copy()

    print("Extracting Last Installment (The Anchor)...")
    # 3. Get the absolute most recent installment for each customer (Pillar 3)
    # Since we sorted chronologically, keeping 'last' gets the most recent row.
    last_inst = df.drop_duplicates(subset=['SK_ID_CURR'], keep='last')[['SK_ID_CURR', 'DAYS_LATE_CLIPPED']]
    last_inst.rename(columns={'DAYS_LATE_CLIPPED': 'LAST_MONTH_LATENESS'}, inplace=True)

    print("Calculating Time Windows...")
    # 4. Create Time Windows (Past 1 Year vs Past 6 Months)
    mask_1yr = (df['DAYS_INSTALMENT'] >= -365) & (df['DAYS_INSTALMENT'] < 0)
    mask_6mo = (df['DAYS_INSTALMENT'] >= -180) & (df['DAYS_INSTALMENT'] < 0)

    # Groupby for 1 Year (Your grading idea)
    agg_1yr = df[mask_1yr].groupby('SK_ID_CURR').agg(
        FREQ_1YR=('IS_LATE', 'mean'),             # % of payments late in last year
        SEV_1YR=('DAYS_LATE_CLIPPED', 'mean')     # Avg days late in last year
    ).reset_index()

    # Groupby for 6 Months
    agg_6mo = df[mask_6mo].groupby('SK_ID_CURR').agg(
        FREQ_6MO=('IS_LATE', 'mean'),             # % of payments late in last 6 months
        SEV_6MO=('DAYS_LATE_CLIPPED', 'mean')     # Avg days late in last 6 months
    ).reset_index()

    print("Merging and Calculating Trends...")
    # 5. Merge everything together onto a master list of unique customers
    base_customers = pd.DataFrame({'SK_ID_CURR': df['SK_ID_CURR'].unique()})
    
    res = base_customers.merge(last_inst, on='SK_ID_CURR', how='left')
    res = res.merge(agg_1yr, on='SK_ID_CURR', how='left')
    res = res.merge(agg_6mo, on='SK_ID_CURR', how='left')

    # Fill NaNs with 0 (For customers who had NO loans in the last year, their stats are 0)
    res.fillna(0, inplace=True)

    # 6. Calculate Trends (The "Slope" Alternative)
    # We add 0.0001 to the denominator to prevent Python from crashing on ZeroDivision errors.
    res['FREQ_ACCELERATION_RATIO'] = res['FREQ_6MO'] / (res['FREQ_1YR'] + 0.0001)
    res['SEV_ACCELERATION_RATIO'] = res['SEV_6MO'] / (res['SEV_1YR'] + 0.0001)

    print("Done!")
    return res

# --- How to run it ---
# time_series_df = build_time_series_features(installments_payments_df)
# print(time_series_df.head())

In [10]:
POS_CASH_balance_df.head(15)

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0
5,2207092,342166,-32,12.0,12.0,Active,0,0
6,1110516,204376,-38,48.0,43.0,Active,0,0
7,1387235,153211,-35,36.0,36.0,Active,0,0
8,1220500,112740,-31,12.0,12.0,Active,0,0
9,2371489,274851,-32,24.0,16.0,Active,0,0


In [11]:
##Now to the POSH CASH BALANCE csv, one of the things we try to catch here is how many times customer failed to pay the 
##installment

In [12]:
import pandas as pd
import numpy as np

def build_advanced_pos_features(df):
    print("Sorting POS data chronologically...")
    # Step 1: Sort by customer and time so we can see the sequence of months
    df = df.sort_values(by=['SK_ID_CURR', 'SK_ID_PREV', 'MONTHS_BALANCE'])

    assert (df['MONTHS_BALANCE'] < 0).all(), "Found MONTHS_BALANCE >= 0 — future data leaking in!"
    
    print("Calculating Month-over-Month changes...")
    # Step 2: Track if the remaining installments decreased from the previous month
    # In a perfect payment, the difference should be -1.
    df['TERM_DIFF'] = df.groupby(['SK_ID_CURR', 'SK_ID_PREV'])['CNT_INSTALMENT_FUTURE'].diff()
    
    # If the difference is 0 and the contract is Active, they missed a payment cycle!
    df['IS_STALLED_MONTH'] = ((df['TERM_DIFF'] == 0) & (df['NAME_CONTRACT_STATUS'] == 'Active')).astype(int)

    print("Building Customer-Level Time Series Summaries...")
    # Step 3: Group everything by the Customer (SK_ID_CURR)
    customer_features = pd.DataFrame({'SK_ID_CURR': df['SK_ID_CURR'].unique()})
    
    # Metric A: Total lifetime stalled payment months (Frozen Debt)
    stalled_counts = df.groupby('SK_ID_CURR')['IS_STALLED_MONTH'].sum().reset_index()
    stalled_counts.rename(columns={'IS_STALLED_MONTH': 'POS_TOTAL_STALLED_MONTHS'}, inplace=True)
    
    # Metric B: Worst lateness in the critical recent window (Last 3 Months)
    mask_recent = df['MONTHS_BALANCE'] >= -3
    recent_dpd = df[mask_recent].groupby('SK_ID_CURR')['SK_DPD'].max().reset_index()
    recent_dpd.rename(columns={'SK_DPD': 'POS_MAX_DAYS_LATE_LAST_3M'}, inplace=True)
    
    # Metric C: Credit Hunger (How many active loans did they have last month?)
    mask_last_month = df['MONTHS_BALANCE'] == -1
    active_loans_last_month = df[mask_last_month & (df['NAME_CONTRACT_STATUS'] == 'Active')].groupby('SK_ID_CURR')['SK_ID_PREV'].nunique().reset_index()
    active_loans_last_month.rename(columns={'SK_ID_PREV': 'POS_ACTIVE_LOANS_COUNT_LAST_MONTH'}, inplace=True)
    
    print("Merging features together...")
    # Combine all your new clever features into one table
    customer_features = customer_features.merge(stalled_counts, on='SK_ID_CURR', how='left')
    customer_features = customer_features.merge(recent_dpd, on='SK_ID_CURR', how='left')
    customer_features = customer_features.merge(active_loans_last_month, on='SK_ID_CURR', how='left')
    
    # Fill empty values with 0 (e.g., if they had no loans last month, count is 0)
    customer_features.fillna(0, inplace=True)
    
    print("Advanced POS Feature Engineering Complete!")
    return customer_features

# --- How to use it in your script ---
# advanced_pos_df = build_advanced_pos_features(POS_CASH_balance_df)

In [13]:
##Now to the lets extract credit card behaviour
##Lets see it

In [18]:
import pandas as pd
import numpy as np

def build_advanced_credit_features(df):
    print("Sorting Credit Card data chronologically...")
    df = df.sort_values(by=['SK_ID_CURR', 'MONTHS_BALANCE'])

    assert (df['MONTHS_BALANCE'] < 0).all(), "Found MONTHS_BALANCE >= 0 — future data leaking in!"
    
    print("Calculating monthly behavioral ratios...")
    # 1. Calculate Utilization (Avoid division by zero by adding 1 to limit)
    df['UTILIZATION_RATE'] = df['AMT_BALANCE'] / (df['AMT_CREDIT_LIMIT_ACTUAL'] + 1)
    
    # 2. Minimum Payment Ratio (Are they just scraping by?)
    # 1.0 means they paid EXACTLY the minimum. Higher means they are paying it off.
    df['PAYMENT_TO_MIN_RATIO'] = df['AMT_PAYMENT_CURRENT'] / (df['AMT_INST_MIN_REGULARITY'] + 0.001)

    print("Building Time Windows...")
    mask_1yr = (df['MONTHS_BALANCE'] >= -12) & (df['MONTHS_BALANCE'] < -12)
    mask_3mo = (df['MONTHS_BALANCE'] >= -3) & (df['MONTHS_BALANCE'] < -12)

    # Groupby for 1 Year baseline
    agg_1yr = df[mask_1yr].groupby('SK_ID_CURR').agg(
        AVG_UTIL_1YR=('UTILIZATION_RATE', 'mean')
    ).reset_index()

    # Groupby for 3 Months recent behavior
    agg_3mo = df[mask_3mo].groupby('SK_ID_CURR').agg(
        AVG_UTIL_3MO=('UTILIZATION_RATE', 'mean'),
        TOTAL_ATM_DRAWINGS_3MO=('AMT_DRAWINGS_ATM_CURRENT', 'sum'),
        AVG_PAY_TO_MIN_3MO=('PAYMENT_TO_MIN_RATIO', 'mean')
    ).reset_index()

    print("Merging and constructing trajectory metrics...")
    customer_features = pd.DataFrame({'SK_ID_CURR': df['SK_ID_CURR'].unique()})
    customer_features = customer_features.merge(agg_1yr, on='SK_ID_CURR', how='left')
    customer_features = customer_features.merge(agg_3mo, on='SK_ID_CURR', how='left')
    
    customer_features.fillna(0, inplace=True)

    # 3. Utilization Acceleration: If > 1, they are loading up the card rapidly right now
    customer_features['CC_UTIL_ACCELERATION_RATIO'] = (
        customer_features['AVG_UTIL_3MO'] / (customer_features['AVG_UTIL_1YR'] + 0.0001)
    )

    # Clean up intermediate columns we don't need to feed to the model
    columns_to_keep = [
        'SK_ID_CURR', 'AVG_UTIL_3MO', 'TOTAL_ATM_DRAWINGS_3MO', 
        'AVG_PAY_TO_MIN_3MO', 'CC_UTIL_ACCELERATION_RATIO'
    ]
    
    print("Advanced Credit Card Feature Engineering Complete!")
    return customer_features[columns_to_keep]

# --- How to run it ---
# advanced_cc_df = build_advanced_credit_features(credit_card_balance_df)

In [19]:
# Load his merged bureau + previous-application features
to_merge_df = pd.read_pickle("dataset/to_merge_df.pkl")

# Build your time features (from your module)
ts_installments = build_time_series_features(installments_payments_df)
ts_pos = build_advanced_pos_features(POS_CASH_balance_df)
ts_cc = build_advanced_credit_features(credit_card_balance_df)

time_features_df = ts_installments.merge(ts_pos, on="SK_ID_CURR", how="outer")
time_features_df = time_features_df.merge(ts_cc, on="SK_ID_CURR", how="outer")

# Concatenate onto his features — one merge, done
to_merge_df = pd.merge(to_merge_df, time_features_df, on="SK_ID_CURR", how="outer")

Sorting data chronologically...
Extracting Last Installment (The Anchor)...
Calculating Time Windows...
Merging and Calculating Trends...
Done!
Sorting POS data chronologically...
Calculating Month-over-Month changes...
Building Customer-Level Time Series Summaries...
Merging features together...
Advanced POS Feature Engineering Complete!
Sorting Credit Card data chronologically...
Calculating monthly behavioral ratios...
Building Time Windows...
Merging and constructing trajectory metrics...
Advanced Credit Card Feature Engineering Complete!


In [21]:
applications_df = pd.read_csv(os.path.join(dataset_dir, "application_train.csv"))

In [22]:
from sklearn.model_selection import train_test_split

applications_train, applications_test = train_test_split(
    applications_df,
    train_size=0.7,
    random_state=42,
    stratify=applications_df["TARGET"]
)

applications_train = applications_train.reset_index(drop=True)
applications_test = applications_test.reset_index(drop=True)

del applications_df
gc.collect()

823

In [25]:
# Process the training set
applications_train = pd.merge(applications_train, to_merge_df, on="SK_ID_CURR", how="left")
applications_train = pd.get_dummies(applications_train)
applications_train = remove_nan_columns(applications_train)

In [26]:
# Process the test set
applications_test = pd.merge(applications_test, to_merge_df, on="SK_ID_CURR", how="left")
applications_test = pd.get_dummies(applications_test)

# Align test columns to match train (important — do this every time, since your
# new time features may introduce column-count mismatches between the two)
applications_test = applications_test.reindex(columns=applications_train.columns, fill_value=0)

In [27]:
from sklearn.impute import SimpleImputer

imp_mean = SimpleImputer(strategy="mean").set_output(transform="pandas")
imp_mean.fit(applications_train)
applications_train = imp_mean.transform(applications_train)
applications_test = imp_mean.transform(applications_test)

In [28]:
applications_train.to_csv(os.path.join(dataset_dir, "train_set.csv"), index=False)
applications_test.to_csv(os.path.join(dataset_dir, "test_set.csv"), index=False)